# Diagnostic: Tail of the Probability Distribution — 2018

**Objective:** Explore what is happening in the right tail of the P(win) distribution by ballot order in 2018, where an anomalous spike is observed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Load data
# ============================================================
# ADJUST PATH according to your machine:
filepath = "../BD/resultados_distrital.xlsx"

df = pd.read_excel(filepath)
df.columns = df.columns.str.strip().str.lower()
if "año" in df.columns:
    df = df.rename(columns={"año": "year"})

# Winner variable
df["max_votos"] = df.groupby(["year", "ubigeo"])["total_votos"].transform("max")
df["winner"] = (df["total_votos"] == df["max_votos"]).astype(int)

# N candidates per district-year
df["n_candidates"] = df.groupby(["year", "ubigeo"])["ubigeo"].transform("count")

print(f"Rows: {len(df):,}")
print(f"Years: {sorted(df['year'].unique())}")
print(f"Districts 2018: {df[df['year']==2018]['ubigeo'].nunique():,}")
print(f"Districts 2022: {df[df['year']==2022]['ubigeo'].nunique():,}")

## 1. Comparative Plot: P(win) by Ballot Order — 2018 vs 2022

We reproduce the scatter plot that shows the problem.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

for idx, year in enumerate([2018, 2022]):
    dfy = df[df["year"] == year]
    prob = dfy.groupby("orden_aparicion")["winner"].agg(["mean", "count"]).reset_index()
    prob.columns = ["order", "prob", "n"]

    ax = axes[idx]
    colors = ["red" if n < 20 else "steelblue" for n in prob["n"]]
    ax.bar(prob["order"], prob["prob"], color=colors, edgecolor="navy", alpha=0.7)

    for _, r in prob.iterrows():
        ax.annotate(
            f'n={int(r["n"])}',
            (r["order"], r["prob"] + 0.01),
            fontsize=6, ha="center", rotation=45,
        )

    ax.set_title(f"{year}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Ballot order")
    ax.set_ylabel("P(win)")
    ax.grid(axis="y", linestyle="--", alpha=0.3)

fig.suptitle(
    "Probability of winning by ballot order\n(red = fewer than 20 obs)",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.show()

## 2. Zoom on the 2018 Tail (order >= 10)

The spike at order = 20 stands out: **3 out of 5 candidates win (60%)**.

In [ ]:
dfy18 = df[df["year"] == 2018]
prob18 = (
    dfy18.groupby("orden_aparicion")["winner"]
    .agg(["mean", "count", "sum"])
    .reset_index()
)
prob18.columns = ["order", "prob", "n_obs", "n_winners"]

tail = prob18[prob18["order"] >= 10].copy()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(
    tail["order"], tail["prob"],
    color=["red" if o >= 18 else "orange" for o in tail["order"]],
    edgecolor="navy", alpha=0.7,
)
for _, r in tail.iterrows():
    ax.annotate(
        f'n={int(r["n_obs"])}\nwin={int(r["n_winners"])}',
        (r["order"], r["prob"] + 0.015),
        fontsize=8, ha="center",
    )

ax.set_title(
    "2018 — ZOOM tail (order >= 10)\nSpike at order 20: 3/5 win = 60%",
    fontsize=13, fontweight="bold", color="red",
)
ax.set_xlabel("Ballot order")
ax.set_ylabel("P(win)")
ax.set_xticks(range(10, int(tail["order"].max()) + 1))
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

print("Full tail table:")
print(tail.to_string(index=False))

## 3. Who Are the Winners in the Tail?

We identify exactly which parties and districts generate the spike.

In [ ]:
# Winners in positions 18, 19, and 20 in 2018
tail_winners = dfy18[
    (dfy18["orden_aparicion"] >= 18) & (dfy18["winner"] == 1)
].copy()

print(f"Winners with order >= 18 in 2018: {len(tail_winners)}")
print()
print(
    tail_winners[
        ["ubigeo", "region", "provincia", "distrito",
         "organizacion_politica", "total_votos", "orden_aparicion"]
    ].sort_values("orden_aparicion").to_string(index=False)
)

In [ ]:
# Summary: which party are they from?
print("Party of tail winners (order >= 18):")
print(tail_winners["organizacion_politica"].value_counts())
print()
print("Region of tail winners (order >= 18):")
print(tail_winners["region"].value_counts())

## 4. Context: Why Does This Happen?

In 2018, some large districts (especially in **Lima**) had up to **20-23 parties** competing. **Accion Popular**, a historic national party with a strong presence in Lima, ended up in high ballot positions (18-20) by lottery, but won anyway thanks to its loyal voter base.

**They did not win BECAUSE of their position; they won DESPITE being in position 20.** Since there are very few observations at those positions (5-19), those few cases of AP in Lima distort the entire average.

Let us see how many candidates those districts had:

In [ ]:
# Districts with 18+ candidates in 2018
cand_count = (
    dfy18.groupby(["ubigeo", "region", "provincia", "distrito"])
    .size()
    .reset_index(name="n_candidates")
)

big_districts = cand_count[cand_count["n_candidates"] >= 18].sort_values(
    "n_candidates", ascending=False
)

print(f"Districts with 18+ candidates in 2018: {len(big_districts)}")
print()
print(big_districts.to_string(index=False))

In [ ]:
# Distribution of number of candidates per district
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, year in enumerate([2018, 2022]):
    dfy = df[df["year"] == year]
    nc = dfy.groupby("ubigeo").size()

    ax = axes[idx]
    ax.hist(nc, bins=range(1, nc.max() + 2), color="steelblue", edgecolor="navy", alpha=0.7)
    ax.set_title(f"{year}: N candidates per district", fontweight="bold")
    ax.set_xlabel("N candidates")
    ax.set_ylabel("N districts")
    ax.axvline(x=15, color="red", linestyle="--", alpha=0.5, label="cutoff = 15")
    ax.legend()

plt.tight_layout()
plt.show()

print("\n2018: n_candidates stats")
print(df[df["year"]==2018].groupby("ubigeo").size().describe())
print("\n2022: n_candidates stats")
print(df[df["year"]==2022].groupby("ubigeo").size().describe())

## 5. All Observations in the Tail (order >= 15, 2018)

In [ ]:
full_tail = dfy18[dfy18["orden_aparicion"] >= 15].copy()

print(f"Total obs with order >= 15 in 2018: {len(full_tail)}")
print(f"Winners: {full_tail['winner'].sum()}")
print()

# By region
print("Distribution by region (tail order >= 15):")
print(full_tail["region"].value_counts())
print()

# Winners by party
print("Winners in the tail by party:")
print(
    full_tail[full_tail["winner"] == 1]["organizacion_politica"]
    .value_counts()
)

## 6. Effect on the Regression: With and Without the Tail

We compare the first stage by truncating the sample at different cutoff points.

In [ ]:
import statsmodels.formula.api as smf

print("Regression: winner ~ orden_aparicion (pooled 2018+2022)")
print("=" * 60)

for max_order in [None, 20, 15, 12, 10]:
    if max_order is None:
        sub = df.copy()
        label = "All obs"
    else:
        sub = df[df["orden_aparicion"] <= max_order].copy()
        label = f"order <= {max_order}"

    mod = smf.ols("winner ~ orden_aparicion", data=sub).fit(cov_type="HC1")
    b = mod.params["orden_aparicion"]
    se = mod.bse["orden_aparicion"]
    t = mod.tvalues["orden_aparicion"]
    p = mod.pvalues["orden_aparicion"]
    n = int(mod.nobs)

    print(f"  {label:20s}  N={n:>6,}  beta={b:>9.5f}  se={se:.5f}  t={t:>7.2f}  p={p:.4f}")

print()
print("2018 only:")
print("-" * 60)
for max_order in [None, 20, 15, 12, 10]:
    sub = dfy18.copy()
    label = "All"
    if max_order is not None:
        sub = sub[sub["orden_aparicion"] <= max_order]
        label = f"order <= {max_order}"

    mod = smf.ols("winner ~ orden_aparicion", data=sub).fit(cov_type="HC1")
    b = mod.params["orden_aparicion"]
    se = mod.bse["orden_aparicion"]
    t = mod.tvalues["orden_aparicion"]
    p = mod.pvalues["orden_aparicion"]
    n = int(mod.nobs)

    print(f"  {label:20s}  N={n:>6,}  beta={b:>9.5f}  se={se:.5f}  t={t:>7.2f}  p={p:.4f}")

## 7. Summary and Conclusion

| Finding | Detail |
|---|---|
| **What is observed** | Spike in P(win) at ballot positions 18-20 in 2018 |
| **Cause** | Accion Popular (a major party) ended up in high ballot positions by lottery in Lima districts with 20+ candidates, but won thanks to its loyal voter base |
| **Magnitude** | Order=20 has only 5 obs, 3 winners (60%); order=18 has 19 obs, 2 winners |
| **Impact** | Attenuates the negative coefficient of order->P(win), weakens the IV first stage |
| **In 2022** | Does not occur: maximum 16 candidates and the tail drops to 0 cleanly |